# GOV-01 V2-E8 — EfficientNetB0 low-learning-rate continuation

**Purpose:** continue the best V2-E7 EfficientNetB0 checkpoint because its validation loss was still improving at the final E7 epoch. E8 keeps the same data, class priorities, and fine-tuned upper layers, but lowers the learning rate from 0.00001 to 0.000005 for smaller, more precise adjustments.

This is one controlled continuation. No image, label, split, augmentation, or protected-test access is changed.

- Train and validation only; protected test is deliberately never loaded.
- V2-E7 remains unchanged. V2-E8 writes to its own Google Drive folder.
- Run the training cell once. After a disconnect, rerun Cells 1–4, then run Cells 6–7; do not retrain.

In [ ]:
# Cell 1 — imports and Google Drive connection
from pathlib import Path
import json
import shutil
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive, files
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support

drive.mount('/content/drive')
print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

In [ ]:
# Cell 2 — fixed paths and approved V2 class order
DRIVE_ROOT = Path('/content/drive/MyDrive/GOV-01')
ARCHIVE_PATH = DRIVE_ROOT / 'v2' / 'multiclass_final.zip'
E7_CHECKPOINT_PATH = DRIVE_ROOT / 'v2_e7_checkpoint' / 'v2_e7_efficientnetb0_finetuned_best.keras'
PERSISTENT_DIR = DRIVE_ROOT / 'v2_e8_checkpoint'

DATA_DIR = Path('/content/data/processed/v2/multiclass_final')
OUTPUT_DIR = Path('/content/v2_e8_output')
CHECKPOINT_PATH = PERSISTENT_DIR / 'v2_e8_efficientnetb0_low_lr_best.keras'
HISTORY_PATH = PERSISTENT_DIR / 'v2_e8_training_history.csv'
SUMMARY_PATH = PERSISTENT_DIR / 'v2_e8_training_summary.json'

CLASS_NAMES = ['crack', 'manhole_cover', 'normal_asphalt', 'pothole', 'repaired_road', 'speed_bump', 'unpaved_road']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
LEARNING_RATE = 5e-6
TARGET_CLASS_WEIGHTS = {'crack': 1.20, 'pothole': 1.60, 'repaired_road': 1.60}
class_weight = {index: TARGET_CLASS_WEIGHTS.get(class_name, 1.00) for index, class_name in enumerate(CLASS_NAMES)}

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Missing V2 dataset ZIP in Google Drive: {ARCHIVE_PATH}')
if not E7_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Missing saved V2-E7 checkpoint: {E7_CHECKPOINT_PATH}')
print('Dataset ZIP:', ARCHIVE_PATH)
print('Starting model:', E7_CHECKPOINT_PATH)
print('V2-E8 learning rate:', LEARNING_RATE)

In [ ]:
# Cell 3 — extract the persistent dataset into the temporary Colab runtime only when needed.
if not (DATA_DIR / 'train').is_dir() or not (DATA_DIR / 'validation').is_dir():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
    print('Extracted dataset to:', DATA_DIR)
else:
    print('Dataset already available in this runtime:', DATA_DIR)
for split_name, expected_count in [('train', 9452), ('validation', 1658)]:
    split_path = DATA_DIR / split_name
    actual_count = sum(1 for path in split_path.rglob('*') if path.is_file())
    if actual_count != expected_count:
        raise RuntimeError(f'{split_name} has {actual_count} files; expected {expected_count}. Stop and inspect the archive.')
    print(f'{split_name}: {actual_count} images')
if (DATA_DIR / 'protected_test').exists():
    print('Protected test is present but intentionally not loaded or evaluated.')

In [ ]:
# Cell 4 — same approved V2 data loaders.
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train', label_mode='int', class_names=CLASS_NAMES, image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
validation_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'validation', label_mode='int', class_names=CLASS_NAMES, image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False
)
if train_ds.class_names != CLASS_NAMES or validation_ds.class_names != CLASS_NAMES:
    raise RuntimeError('Class order differs from the approved V2 label order.')
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)
print('Class order:', CLASS_NAMES)
print('Protected test remains unused.')

In [ ]:
# Cell 5 — continue the saved E7 model with smaller learning steps. Run once.
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model = tf.keras.models.load_model(E7_CHECKPOINT_PATH)
base_candidates = [layer for layer in model.layers if isinstance(layer, tf.keras.Model) and layer.name.startswith('efficientnetb0')]
if len(base_candidates) != 1:
    raise RuntimeError(f'Expected one EfficientNetB0 base model, found {len(base_candidates)}.')
base_model = base_candidates[0]
UNFREEZE_LAST_LAYERS = 30
fine_tune_start = max(0, len(base_model.layers) - UNFREEZE_LAST_LAYERS)
base_model.trainable = True
for layer in base_model.layers[:fine_tune_start]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_start:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.CSVLogger(HISTORY_PATH, append=False),
]
print('Starting from V2-E7 with the same 30 fine-tuned layers and smaller learning steps.')
start_time = time.time()
history = model.fit(train_ds, validation_data=validation_ds, epochs=12, class_weight=class_weight, callbacks=callbacks, verbose=1)
training_seconds = time.time() - start_time
summary = {
    'experiment': 'V2-E8', 'backbone': 'EfficientNetB0', 'starting_checkpoint': str(E7_CHECKPOINT_PATH),
    'learning_rate': LEARNING_RATE, 'unfreeze_last_layers': UNFREEZE_LAST_LAYERS,
    'target_class_weights': TARGET_CLASS_WEIGHTS, 'epochs_completed': len(history.history['loss']),
    'training_seconds': training_seconds, 'best_validation_loss': float(min(history.history['val_loss'])),
    'best_validation_accuracy': float(max(history.history['val_accuracy'])),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Training completed in {training_seconds:.1f} seconds.')
print('Best V2-E8 model saved:', CHECKPOINT_PATH)

In [ ]:
# Cell 6 — validate the saved best V2-E8 checkpoint. After disconnect, rerun Cells 1–4 first.
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'No completed V2-E8 checkpoint found in Drive: {CHECKPOINT_PATH}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model = tf.keras.models.load_model(CHECKPOINT_PATH)
y_validation = np.concatenate([labels.numpy() for _, labels in validation_ds])
probabilities = best_model.predict(validation_ds, verbose=1)
predictions = probabilities.argmax(axis=1)
precision, recall, f1_values, support = precision_recall_fscore_support(y_validation, predictions, labels=np.arange(len(CLASS_NAMES)), zero_division=0)
e8_metrics = {
    'experiment': 'V2-E8', 'backbone': 'EfficientNetB0', 'split': 'validation', 'protected_test_used': False,
    'image_count': int(len(y_validation)), 'accuracy': float(accuracy_score(y_validation, predictions)),
    'macro_f1': float(f1_score(y_validation, predictions, average='macro')),
    'per_class': {class_name: {'precision': float(precision[index]), 'recall': float(recall[index]), 'f1': float(f1_values[index]), 'support': int(support[index])} for index, class_name in enumerate(CLASS_NAMES)},
}
(OUTPUT_DIR / 'v2_e8_validation_metrics.json').write_text(json.dumps(e8_metrics, indent=2), encoding='utf-8')
comparison = {
    'split': 'validation', 'protected_test_used': False,
    'V2-E1_compact_cnn': {'accuracy': 0.6984318455971049, 'macro_f1': 0.6795904943261485},
    'V2-E4_targeted_class_weights': {'accuracy': 0.8642943305186972, 'macro_f1': 0.8803169234751674},
    'V2-E7_efficientnetb0': {'accuracy': 0.8661037394451147, 'macro_f1': 0.8854298526310159},
    'V2-E8_efficientnetb0_low_lr': {'accuracy': e8_metrics['accuracy'], 'macro_f1': e8_metrics['macro_f1']},
}
(OUTPUT_DIR / 'v2_e8_comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
report = classification_report(y_validation, predictions, target_names=CLASS_NAMES, digits=4, zero_division=0)
(OUTPUT_DIR / 'v2_e8_validation_classification_report.txt').write_text(report, encoding='utf-8')
print('Validation accuracy:', f"{e8_metrics['accuracy']:.4f}")
print('Validation macro F1:', f"{e8_metrics['macro_f1']:.4f}")
print(report)
matrix = confusion_matrix(y_validation, predictions, labels=np.arange(len(CLASS_NAMES)))
fig, axis = plt.subplots(figsize=(10, 8)); image = axis.imshow(matrix, cmap='Blues')
axis.set_xticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha='right'); axis.set_yticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES)
axis.set_xlabel('Predicted label'); axis.set_ylabel('Actual label'); axis.set_title('V2-E8 validation confusion matrix')
threshold = matrix.max() / 2 if matrix.max() else 0
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(column, row, str(matrix[row, column]), ha='center', va='center', color='white' if matrix[row, column] > threshold else 'black')
fig.colorbar(image, ax=axis); fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'v2_e8_validation_confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# Cell 7 — learning curves and browser-download package. Do not commit the package or model.
if not HISTORY_PATH.is_file():
    raise FileNotFoundError(f'Missing V2-E8 training history in Drive: {HISTORY_PATH}')
history_frame = pd.read_csv(HISTORY_PATH)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_frame['epoch'] + 1, history_frame['accuracy'], label='train'); axes[0].plot(history_frame['epoch'] + 1, history_frame['val_accuracy'], label='validation')
axes[0].set_title('V2-E8 accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()
axes[1].plot(history_frame['epoch'] + 1, history_frame['loss'], label='train'); axes[1].plot(history_frame['epoch'] + 1, history_frame['val_loss'], label='validation')
axes[1].set_title('V2-E8 loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'v2_e8_learning_curve.png', dpi=150); plt.show()
for source_path in [CHECKPOINT_PATH, HISTORY_PATH, SUMMARY_PATH]:
    if source_path.is_file():
        shutil.copy2(source_path, OUTPUT_DIR / source_path.name)
archive_path = shutil.make_archive('/content/v2_e8_output', 'zip', OUTPUT_DIR)
print('Created browser-download package:', archive_path)
print('Keep this ZIP outside the Git repository. Do not commit it.')
files.download(archive_path)